In [113]:
import numpy as np
import sympy as sy
import re
import pandas as pd
import ast

In [114]:
df = pd.read_csv('./3x3 file analysis.csv', index_col=0, converters={
   'PairID': int, 
   'Matrix': ast.literal_eval,
   'EigenResult1': sy.sympify,
   'EigenResult2': sy.sympify,
   'EigenResult3': sy.sympify,
   })
print(df.head())

   PairID                             Matrix           EigenResult1  \
1       0  [[1, 1, 1], [1, 1, 1], [9, 9, 9]]  [0, [[-1], [1], [0]]]   
2       0  [[1, 1, 1], [1, 1, 1], [9, 9, 9]]  [0, [[-1], [1], [0]]]   
3       1  [[1, 1, 1], [2, 1, 2], [8, 9, 8]]  [0, [[-1], [0], [1]]]   
4       1  [[1, 1, 1], [1, 6, 1], [9, 4, 9]]  [0, [[-1], [0], [1]]]   
5       2  [[1, 1, 1], [5, 1, 5], [5, 9, 5]]  [0, [[-1], [0], [1]]]   

             EigenResult2                  EigenResult3  
1   [0, [[-1], [0], [1]]]     [11, [[1/9], [1/9], [1]]]  
2   [0, [[-1], [0], [1]]]     [11, [[1/9], [1/9], [1]]]  
3  [-1, [[0], [-1], [1]]]  [11, [[6/49], [11/49], [1]]]  
4   [5, [[0], [-1], [1]]]  [11, [[6/49], [11/49], [1]]]  
5  [-4, [[0], [-1], [1]]]  [11, [[3/19], [11/19], [1]]]  


In [115]:
# Restructure the data
def eigenSplit (dataframe, column):
    EVal = [x[0] for x in dataframe[column].tolist()]
    EVect = [x[1] for x in dataframe[column].tolist()]
    EVect = [[x[0][0],x[1][0],x[2][0]] for x in EVect]
    return [EVal, EVect]

df2 = df.copy(deep=True)

In [116]:
[EVal1, EVect1] = eigenSplit(df2,'EigenResult1')
[EVal2, EVect2] = eigenSplit(df2,'EigenResult2')
[EVal3, EVect3] = eigenSplit(df2,'EigenResult3')

df2.loc[:,'Eigenvalue1'] = EVal1
df2.loc[:,'Eigenvector1'] = EVect1
df2.loc[:,'Eigenvalue2'] = EVal2
df2.loc[:,'Eigenvector2'] = EVect2
df2.loc[:,'Eigenvalue3'] = EVal3
df2.loc[:,'Eigenvector3'] = EVect3

df2.drop(columns = ['EigenResult1','EigenResult2','EigenResult3'], inplace=True)

print(df2.head())
print(df2.loc[57])

   PairID                             Matrix Eigenvalue1 Eigenvector1  \
1       0  [[1, 1, 1], [1, 1, 1], [9, 9, 9]]           0   [-1, 1, 0]   
2       0  [[1, 1, 1], [1, 1, 1], [9, 9, 9]]           0   [-1, 1, 0]   
3       1  [[1, 1, 1], [2, 1, 2], [8, 9, 8]]           0   [-1, 0, 1]   
4       1  [[1, 1, 1], [1, 6, 1], [9, 4, 9]]           0   [-1, 0, 1]   
5       2  [[1, 1, 1], [5, 1, 5], [5, 9, 5]]           0   [-1, 0, 1]   

  Eigenvalue2 Eigenvector2 Eigenvalue3      Eigenvector3  
1           0   [-1, 0, 1]          11     [1/9, 1/9, 1]  
2           0   [-1, 0, 1]          11     [1/9, 1/9, 1]  
3          -1   [0, -1, 1]          11  [6/49, 11/49, 1]  
4           5   [0, -1, 1]          11  [6/49, 11/49, 1]  
5          -4   [0, -1, 1]          11  [3/19, 11/19, 1]  
PairID                                                         28
Matrix                          [[1, 1, 1], [4, 1, 6], [2, 3, 2]]
Eigenvalue1     4/3 + (-1/2 - sqrt(3)*I/2)*(379/27 + 8*sqrt(42...
Eigenvect

In [ ]:
EV1 = df2['Eigenvector1'].to_list()
EV2 = df2['Eigenvector2'].to_list()
EV3 = df2['Eigenvector3'].to_list()

print('data lists pulled')

print('start EV1')
EV1Bool = [all([y.is_real for y in x]) for x in EV1]
print('start EV2')
EV2Bool = [all([y.is_real for y in x]) for x in EV2]
print('start EV3')
EV3Bool = [all([y.is_real for y in x]) for x in EV3]
print('complete')

data lists pulled
start EV1
start EV2
start EV3
complete


In [308]:
combined = [a and b and c for a,b,c in zip(EV1Bool, EV2Bool, EV3Bool)]
df3 = df2.loc[combined]

In [312]:
from collections import Counter

# Checking if any pairings only had one imaginary output
pairs = df3['PairID'].tolist()
count = Counter(pairs)

if 1 in count.values():
    print('Found non-paired PairID')
    keys = [key for key, value in count.items() if value==1]
    print(keys)
else:
    print('All PairIDs are in pairs')


All PairIDs are in pairs


In [ ]:
# Imaginary numbers have been filtered out within df3
# Now to compare paired values

IDs = list(set(df3['PairID'].tolist()))
Vectors = [df3.loc[df['PairID'] == x, ['PairID', 'Eigenvector1', 'Eigenvector2', 'Eigenvector3']] for x in IDs]

   PairID Eigenvector1 Eigenvector2   Eigenvector3
1       0   [-1, 1, 0]   [-1, 0, 1]  [1/9, 1/9, 1]
2       0   [-1, 1, 0]   [-1, 0, 1]  [1/9, 1/9, 1]


In [345]:
Eigenvectors1 = [Vectors[x]['Eigenvector1'].tolist() for x in range(1,len(Vectors))]
Eigenvectors2 = [Vectors[x]['Eigenvector2'].tolist() for x in range(1,len(Vectors))]
Eigenvectors3 = [Vectors[x]['Eigenvector3'].tolist() for x in range(1,len(Vectors))]

In [358]:
def checkEquality (arr):
    for x in range(0,len(arr)):
        for y in range(0,2):
            if arr[x][0][y].equals(arr[x][1][y]):
                continue
            else:
                print(f'Failed: {x}')

checkEquality(Eigenvectors1)

Failed: 393
Failed: 450
Failed: 452
Failed: 462
Failed: 501
Failed: 533
Failed: 571
Failed: 571
Failed: 578
Failed: 586
Failed: 587
Failed: 588
Failed: 588
Failed: 682
Failed: 1013
Failed: 1122
Failed: 1183
Failed: 1184
Failed: 1185
Failed: 1315
Failed: 1511
Failed: 1512
Failed: 1513
Failed: 1603
Failed: 1604
Failed: 1605
Failed: 1606
Failed: 1607
Failed: 1608
Failed: 1694
Failed: 1695
Failed: 1696
Failed: 1723
Failed: 1774
Failed: 1775
Failed: 1776
Failed: 1778
Failed: 1778
Failed: 1785
Failed: 1793
Failed: 1794
Failed: 1795
Failed: 1795
Failed: 1915
Failed: 2027
Failed: 2027
Failed: 2029
Failed: 2029
Failed: 2031
Failed: 2031
Failed: 2230
Failed: 2239
Failed: 2245
Failed: 2277
Failed: 2298
Failed: 2300
Failed: 2345
Failed: 2424
Failed: 2430
Failed: 2438
Failed: 2489
Failed: 2835
Failed: 2835
Failed: 2837
Failed: 2837
Failed: 2839
Failed: 2839
Failed: 2879
Failed: 2891
Failed: 2978
Failed: 2978
Failed: 2980
Failed: 3406
Failed: 3414
Failed: 3460
Failed: 3464
Failed: 3477
Failed: 3569


In [359]:
print(Eigenvectors1[393])
print(Eigenvectors2[393])
print(Eigenvectors3[393])

[[-1, 0, 1], [-1, 1, 0]]
[[-1, 0, 1], [-1, 1, 0]]
[[11/24, 7/24, 1], [11/24, 7/24, 1]]
